# WorldCup2026 승부예측

목표:
지난 1년간 경기만 모델 학습/검증에 사용

입력 데이터:
1. Transfermarkt 국가 전력
2. FotMob 최근 5경기 평균 스탯

중요:
최근 5경기 평균은 현재 경기 제외
즉 shift(1) 사용


# 1단계: 라이브러리 불러오기

In [93]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import unicodedata


# 2단계: 경로 설정

In [94]:
PROJECT_ROOT = Path.cwd()

# 만약 현재 위치가 WorldCup2026 폴더가 아니면 직접 지정
# PROJECT_ROOT = Path("/Users/minseobeom/Desktop/WorldCup2026")

DATA_DIR = PROJECT_ROOT / "data" / "processed"

matches_path = DATA_DIR / "fotmob_match_ids_extracted.csv"
stats_path = DATA_DIR / "fotmob_statistics_long.csv"
transfermarkt_path = DATA_DIR / "transfermarkt_country_profile.csv"

print(DATA_DIR)


/Users/minseobeom/Desktop/WorldCup2026/data/processed


# 3단계: 함수 만들기

In [95]:
def normalize_team_name(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def to_number(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).replace(",", "")
    match = re.search(r"-?\d+(\.\d+)?", text)

    if match:
        return float(match.group(0))

    return np.nan


# 4단계: 경기 데이터 불러오기

In [96]:
matches = pd.read_csv(matches_path)

matches["date"] = pd.to_datetime(matches["date"])
matches["home_team"] = matches["home_team"].apply(normalize_team_name)
matches["away_team"] = matches["away_team"].apply(normalize_team_name)

matches["fotmob_match_id"] = pd.to_numeric(matches["fotmob_match_id"], errors="coerce")

matches = matches.dropna(subset=[
    "fotmob_match_id",
    "home_score",
    "away_score"
]).copy()

matches["fotmob_match_id"] = matches["fotmob_match_id"].astype(int)

matches = matches.sort_values(["date", "fotmob_match_id"]).reset_index(drop=True)

print(matches.shape)
matches.head()


(1514, 26)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,...,fotmob_home_long,fotmob_home_score,fotmob_away_team,fotmob_away_long,fotmob_away_score,fotmob_status,fotmob_utc_time,match_score,match_reversed,fotmob_match_status
0,2023-01-06,Iraq,Oman,0,0,Gulf Cup,Basra,Iraq,False,draw,...,Iraq,0.0,Oman,Oman,0.0,FT,2023-01-06T16:00:00.000Z,1.08,False,matched
1,2023-01-06,Yemen,Saudi Arabia,0,2,Gulf Cup,Basra,Iraq,True,away_win,...,Yemen,0.0,Saudi Arabia,Saudi Arabia,2.0,FT,2023-01-06T18:45:00.000Z,1.08,False,matched
2,2023-01-07,Kuwait,Qatar,0,2,Gulf Cup,Basra,Iraq,True,away_win,...,Kuwait,0.0,Qatar,Qatar,2.0,FT,2023-01-07T16:15:00.000Z,1.08,False,matched
3,2023-01-09,Iraq,Saudi Arabia,2,0,Gulf Cup,Basra,Iraq,False,home_win,...,Saudi Arabia,0.0,Iraq,Iraq,2.0,FT,2023-01-09T16:15:00.000Z,1.08,True,matched
4,2023-01-09,Sweden,Finland,2,0,Friendly,Faro-Loulé,Portugal,True,home_win,...,Sweden,2.0,Finland,Finland,0.0,FT,2023-01-09T18:45:00.000Z,1.08,False,matched


# 5단계: 지난 1년 기준 날짜 만들기

In [97]:
max_date = matches["date"].max()
start_date = max_date - pd.DateOffset(years=1)

print("데이터 마지막 날짜:", max_date.date())
print("지난 1년 시작 날짜:", start_date.date())


데이터 마지막 날짜: 2026-06-10
지난 1년 시작 날짜: 2025-06-10


# 6단계: FotMob 통계 불러오기

In [98]:
stats = pd.read_csv(stats_path)

stats.head()


,fotmob_match_id,period,group_title,group_key,stat_title,stat_key,home_value,away_value,format,type,highlighted
0,4071359,All,Top stats,top_stats,Ball possession,BallPossesion,54,46,integer,graph,home
1,4071359,All,Top stats,top_stats,Total shots,total_shots,15,5,integer,text,home
2,4071359,All,Top stats,top_stats,Shots on target,ShotsOnTarget,5,1,integer,text,home
3,4071359,All,Top stats,top_stats,Touches in opposition box,touches_opp_box,22,9,NaN,text,home
4,4071359,All,Top stats,top_stats,Big chances,big_chance,1,1,integer,text,equal


# 7단계: 사용할 FotMob 스탯 선택

In [99]:
use_stats = [
    "BallPossesion",
    "total_shots",
    "ShotsOnTarget",
    "expected_goals",
    "accurate_passes",
    "corners",
    "yellow_cards",
    "red_cards",
    "fouls",
    "keeper_saves",
    "interceptions",
    "clearances",
    "duel_won",
    "touches_opp_box",
]

stats = stats[
    (stats["period"] == "All") &
    (stats["stat_key"].isin(use_stats))
].copy()

stats["home_value_num"] = stats["home_value"].apply(to_number)
stats["away_value_num"] = stats["away_value"].apply(to_number)

print(stats.shape)
stats.head()


(25354, 13)


,fotmob_match_id,period,group_title,group_key,stat_title,stat_key,home_value,away_value,format,type,highlighted,home_value_num,away_value_num
0,4071359,All,Top stats,top_stats,Ball possession,BallPossesion,54,46,integer,graph,home,54.0,46.0
1,4071359,All,Top stats,top_stats,Total shots,total_shots,15,5,integer,text,home,15.0,5.0
2,4071359,All,Top stats,top_stats,Shots on target,ShotsOnTarget,5,1,integer,text,home,5.0,1.0
3,4071359,All,Top stats,top_stats,Touches in opposition box,touches_opp_box,22,9,NaN,text,home,22.0,9.0
6,4071359,All,Top stats,top_stats,Accurate passes,accurate_passes,465 (87%),372 (83%),integerWithPercentage,text,home,465.0,372.0


# 단계: FotMob 스탯을 넓은 형태로 변환

In [100]:
home_stats = stats[[
    "fotmob_match_id",
    "stat_key",
    "home_value_num"
]].rename(columns={"home_value_num": "value"})

away_stats = stats[[
    "fotmob_match_id",
    "stat_key",
    "away_value_num"
]].rename(columns={"away_value_num": "value"})

home_stats_wide = home_stats.pivot_table(
    index="fotmob_match_id",
    columns="stat_key",
    values="value",
    aggfunc="first"
).add_prefix("stat_").reset_index()

away_stats_wide = away_stats.pivot_table(
    index="fotmob_match_id",
    columns="stat_key",
    values="value",
    aggfunc="first"
).add_prefix("stat_").reset_index()

print(home_stats_wide.shape)
home_stats_wide.head()


(1336, 15)


stat_key,fotmob_match_id,stat_BallPossesion,stat_ShotsOnTarget,stat_accurate_passes,stat_clearances,stat_corners,stat_duel_won,stat_expected_goals,stat_fouls,stat_interceptions,stat_keeper_saves,stat_red_cards,stat_total_shots,stat_touches_opp_box,stat_yellow_cards
0,3859879,32.0,3.0,184.0,19.0,4.0,32.0,NaN,11.0,14.0,5.0,0.0,7.0,7.0,2.0
1,3859880,58.0,4.0,415.0,4.0,13.0,54.0,NaN,16.0,9.0,0.0,0.0,16.0,33.0,0.0
2,3859893,65.0,7.0,577.0,13.0,4.0,49.0,NaN,15.0,6.0,0.0,0.0,16.0,51.0,1.0
3,3859898,42.0,3.0,277.0,17.0,2.0,53.0,NaN,22.0,4.0,4.0,0.0,10.0,15.0,1.0
4,3859899,69.0,4.0,501.0,14.0,9.0,48.0,NaN,10.0,10.0,3.0,0.0,25.0,38.0,1.0


# 9단계: 경기 데이터를 팀 기준으로 변환

In [101]:
home_team_rows = matches.copy()

home_team_rows["team"] = home_team_rows["home_team"]
home_team_rows["opponent"] = home_team_rows["away_team"]
home_team_rows["is_home"] = 1
home_team_rows["goals_for"] = home_team_rows["home_score"]
home_team_rows["goals_against"] = home_team_rows["away_score"]

home_team_rows = home_team_rows.merge(
    home_stats_wide,
    on="fotmob_match_id",
    how="left"
)


away_team_rows = matches.copy()

away_team_rows["team"] = away_team_rows["away_team"]
away_team_rows["opponent"] = away_team_rows["home_team"]
away_team_rows["is_home"] = 0
away_team_rows["goals_for"] = away_team_rows["away_score"]
away_team_rows["goals_against"] = away_team_rows["home_score"]

away_team_rows = away_team_rows.merge(
    away_stats_wide,
    on="fotmob_match_id",
    how="left"
)


team_rows = pd.concat([home_team_rows, away_team_rows], ignore_index=True)

team_rows["goal_diff"] = team_rows["goals_for"] - team_rows["goals_against"]

team_rows["points"] = np.select(
    [
        team_rows["goal_diff"] > 0,
        team_rows["goal_diff"] == 0
    ],
    [
        3,
        1
    ],
    default=0
)

team_rows = team_rows.sort_values(["team", "date", "fotmob_match_id"]).reset_index(drop=True)

print(team_rows.shape)
team_rows.head()


(3028, 47)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,...,stat_expected_goals,stat_fouls,stat_interceptions,stat_keeper_saves,stat_red_cards,stat_total_shots,stat_touches_opp_box,stat_yellow_cards,goal_diff,points
0,2023-11-16,Qatar,Afghanistan,8,1,FIFA World Cup qualification,Al Rayyan,Qatar,False,home_win,...,NaN,15.0,8.0,9.0,1.0,2.0,2.0,3.0,-7,0
1,2024-06-06,Afghanistan,Qatar,0,0,FIFA World Cup qualification,Al Hofuf,Saudi Arabia,True,draw,...,NaN,15.0,6.0,2.0,0.0,2.0,10.0,4.0,0,1
2,2023-09-07,Czech Republic,Albania,1,1,UEFA Euro qualification,Prague,Czech Republic,False,draw,...,0.02,9.0,13.0,3.0,0.0,1.0,NaN,2.0,0,1
3,2023-10-12,Albania,Czech Republic,3,0,UEFA Euro qualification,Tirana,Albania,False,home_win,...,1.26,10.0,15.0,7.0,0.0,8.0,12.0,1.0,3,3
4,2024-03-25,Sweden,Albania,1,0,Friendly,Stockholm,Sweden,False,home_win,...,NaN,8.0,12.0,5.0,0.0,8.0,18.0,2.0,-1,0


# 10단계: 최근 5경기 평균 만들기

In [102]:
rolling_cols = [
    "goals_for",
    "goals_against",
    "goal_diff",
    "points",
]

stat_cols = [col for col in team_rows.columns if col.startswith("stat_")]

rolling_cols = rolling_cols + stat_cols

print("최근 5경기 평균에 사용할 컬럼 수:", len(rolling_cols))
print(rolling_cols)


최근 5경기 평균에 사용할 컬럼 수: 18
['goals_for', 'goals_against', 'goal_diff', 'points', 'stat_BallPossesion', 'stat_ShotsOnTarget', 'stat_accurate_passes', 'stat_clearances', 'stat_corners', 'stat_duel_won', 'stat_expected_goals', 'stat_fouls', 'stat_interceptions', 'stat_keeper_saves', 'stat_red_cards', 'stat_total_shots', 'stat_touches_opp_box', 'stat_yellow_cards']


In [103]:
team_rows_rolling = []

for team, group in team_rows.groupby("team"):
    group = group.sort_values(["date", "fotmob_match_id"]).copy()

    # 현재 경기는 제외하고 이전 경기만 사용
    previous_games = group[rolling_cols].shift(1)

    recent5 = previous_games.rolling(
        window=5,
        min_periods=1
    ).mean()

    recent5.columns = [f"recent5_{col}" for col in recent5.columns]

    group = pd.concat([group, recent5], axis=1)

    team_rows_rolling.append(group)

team_rows = pd.concat(team_rows_rolling, ignore_index=True)

recent5_cols = [col for col in team_rows.columns if col.startswith("recent5_")]

print("생성된 recent5 컬럼 수:", len(recent5_cols))
print(recent5_cols[:10])


생성된 recent5 컬럼 수: 18
['recent5_goals_for', 'recent5_goals_against', 'recent5_goal_diff', 'recent5_points', 'recent5_stat_BallPossesion', 'recent5_stat_ShotsOnTarget', 'recent5_stat_accurate_passes', 'recent5_stat_clearances', 'recent5_stat_corners', 'recent5_stat_duel_won']


# 11단계: 홈팀/원정팀 최근 5경기 평균 붙이기

In [104]:
home_recent5 = team_rows[team_rows["is_home"] == 1][
    ["fotmob_match_id", "team"] + recent5_cols
].copy()

home_recent5 = home_recent5.rename(columns={"team": "home_team"})
home_recent5 = home_recent5.rename(columns={
    col: f"home_{col}" for col in recent5_cols
})


away_recent5 = team_rows[team_rows["is_home"] == 0][
    ["fotmob_match_id", "team"] + recent5_cols
].copy()

away_recent5 = away_recent5.rename(columns={"team": "away_team"})
away_recent5 = away_recent5.rename(columns={
    col: f"away_{col}" for col in recent5_cols
})


df = matches.merge(
    home_recent5,
    on=["fotmob_match_id", "home_team"],
    how="left"
)

df = df.merge(
    away_recent5,
    on=["fotmob_match_id", "away_team"],
    how="left"
)

print(df.shape)
df.head()


(1514, 62)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,...,away_recent5_stat_corners,away_recent5_stat_duel_won,away_recent5_stat_expected_goals,away_recent5_stat_fouls,away_recent5_stat_interceptions,away_recent5_stat_keeper_saves,away_recent5_stat_red_cards,away_recent5_stat_total_shots,away_recent5_stat_touches_opp_box,away_recent5_stat_yellow_cards
0,2023-01-06,Iraq,Oman,0,0,Gulf Cup,Basra,Iraq,False,draw,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-06,Yemen,Saudi Arabia,0,2,Gulf Cup,Basra,Iraq,True,away_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-07,Kuwait,Qatar,0,2,Gulf Cup,Basra,Iraq,True,away_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-09,Iraq,Saudi Arabia,2,0,Gulf Cup,Basra,Iraq,False,home_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-09,Sweden,Finland,2,0,Friendly,Faro-Loulé,Portugal,True,home_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 12단계: Transfermarkt 붙이기

In [105]:
tm = pd.read_csv(transfermarkt_path)

tm["country"] = tm["country"].apply(normalize_team_name)

tm = tm[[
    "country",
    "squad_size",
    "average_age",
    "fifa_world_ranking",
    "total_market_value_eur"
]].copy()

for col in [
    "squad_size",
    "average_age",
    "fifa_world_ranking",
    "total_market_value_eur"
]:
    tm[col] = pd.to_numeric(tm[col], errors="coerce")

home_tm = tm.rename(columns={
    "country": "home_team",
    "squad_size": "home_squad_size",
    "average_age": "home_average_age",
    "fifa_world_ranking": "home_fifa_world_ranking",
    "total_market_value_eur": "home_total_market_value_eur"
})

away_tm = tm.rename(columns={
    "country": "away_team",
    "squad_size": "away_squad_size",
    "average_age": "away_average_age",
    "fifa_world_ranking": "away_fifa_world_ranking",
    "total_market_value_eur": "away_total_market_value_eur"
})

df = df.merge(home_tm, on="home_team", how="left")
df = df.merge(away_tm, on="away_team", how="left")

df[[
    "home_team",
    "away_team",
    "home_total_market_value_eur",
    "away_total_market_value_eur"
]].head()


,home_team,away_team,home_total_market_value_eur,away_total_market_value_eur
0,Iraq,Oman,21200000.0,NaN
1,Yemen,Saudi Arabia,NaN,40680000.0
2,Kuwait,Qatar,NaN,19930000.0
3,Iraq,Saudi Arabia,21200000.0,40680000.0
4,Sweden,Finland,406080000.0,NaN


# 13단계: Transfermarkt 차이 만들기

In [106]:
df["diff_squad_size"] = df["home_squad_size"] - df["away_squad_size"]
df["diff_average_age"] = df["home_average_age"] - df["away_average_age"]

# FIFA 랭킹은 낮을수록 강함
df["diff_fifa_world_ranking"] = (
    df["home_fifa_world_ranking"] - df["away_fifa_world_ranking"]
)

df["diff_total_market_value_eur"] = (
    df["home_total_market_value_eur"] - df["away_total_market_value_eur"]
)

df[[
    "home_team",
    "away_team",
    "diff_fifa_world_ranking",
    "diff_total_market_value_eur"
]].head()


,home_team,away_team,diff_fifa_world_ranking,diff_total_market_value_eur
0,Iraq,Oman,NaN,NaN
1,Yemen,Saudi Arabia,NaN,NaN
2,Kuwait,Qatar,NaN,NaN
3,Iraq,Saudi Arabia,-4.0,-19480000.0
4,Sweden,Finland,NaN,NaN


# 14단계: 최근 5경기 차이 만들기

In [107]:
for col in recent5_cols:
    home_col = f"home_{col}"
    away_col = f"away_{col}"
    diff_col = f"diff_{col}"

    df[diff_col] = df[home_col] - df[away_col]

df.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,...,diff_recent5_stat_corners,diff_recent5_stat_duel_won,diff_recent5_stat_expected_goals,diff_recent5_stat_fouls,diff_recent5_stat_interceptions,diff_recent5_stat_keeper_saves,diff_recent5_stat_red_cards,diff_recent5_stat_total_shots,diff_recent5_stat_touches_opp_box,diff_recent5_stat_yellow_cards
0,2023-01-06,Iraq,Oman,0,0,Gulf Cup,Basra,Iraq,False,draw,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-06,Yemen,Saudi Arabia,0,2,Gulf Cup,Basra,Iraq,True,away_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-07,Kuwait,Qatar,0,2,Gulf Cup,Basra,Iraq,True,away_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-09,Iraq,Saudi Arabia,2,0,Gulf Cup,Basra,Iraq,False,home_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-09,Sweden,Finland,2,0,Friendly,Faro-Loulé,Portugal,True,home_win,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 15단계: 정답값 만들기

In [108]:
def make_target(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    elif row["home_score"] < row["away_score"]:
        return "away_win"
    else:
        return "draw"


df["target"] = df.apply(make_target, axis=1)

df[[
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "target"
]].head()


,date,home_team,away_team,home_score,away_score,target
0,2023-01-06,Iraq,Oman,0,0,draw
1,2023-01-06,Yemen,Saudi Arabia,0,2,away_win
2,2023-01-07,Kuwait,Qatar,0,2,away_win
3,2023-01-09,Iraq,Saudi Arabia,2,0,home_win
4,2023-01-09,Sweden,Finland,2,0,home_win


# 16단계: 지난 1년 경기만 필터링

In [109]:
model_df = df[df["date"] >= start_date].copy()

model_df = model_df.sort_values("date").reset_index(drop=True)

print("지난 1년 경기 수:", len(model_df))
print(model_df["date"].min(), "~", model_df["date"].max())
print(model_df["target"].value_counts())


지난 1년 경기 수: 460
2025-06-10 00:00:00 ~ 2026-06-10 00:00:00
target
home_win    225
away_win    128
draw        107
Name: count, dtype: int64


# 17단계: 모델에 사용할 피처 선택

In [110]:
transfermarkt_features = [
    "home_squad_size",
    "away_squad_size",
    "home_average_age",
    "away_average_age",
    "home_fifa_world_ranking",
    "away_fifa_world_ranking",
    "home_total_market_value_eur",
    "away_total_market_value_eur",
    "diff_squad_size",
    "diff_average_age",
    "diff_fifa_world_ranking",
    "diff_total_market_value_eur",
]

recent5_features = []

for col in recent5_cols:
    recent5_features.append(f"home_{col}")
    recent5_features.append(f"away_{col}")
    recent5_features.append(f"diff_{col}")

feature_cols = transfermarkt_features + recent5_features + [
    "neutral"
]

X = model_df[feature_cols]
y = model_df["target"]

print(X.shape)
print(y.value_counts())


(460, 67)
target
home_win    225
away_win    128
draw        107
Name: count, dtype: int64


# 18단계: 시간순 훈련/검증 분리

In [111]:
split_idx = int(len(model_df) * 0.8)

train_df = model_df.iloc[:split_idx].copy()
valid_df = model_df.iloc[split_idx:].copy()

X_train = train_df[feature_cols]
y_train = train_df["target"]

X_valid = valid_df[feature_cols]
y_valid = valid_df["target"]

print("train:", len(train_df))
print("valid:", len(valid_df))
print("검증 시작 날짜:", valid_df["date"].min())


train: 368
valid: 92
검증 시작 날짜: 2026-03-30 00:00:00


# 19단계: 모델 학습

In [112]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=5
    ))
])

model.fit(X_train, y_train)


,steps,"[('imputer', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,n_estimators,300


# 20단계: 성능 확인

In [113]:
from sklearn.metrics import accuracy_score, classification_report, log_loss

pred = model.predict(X_valid)
proba = model.predict_proba(X_valid)

print("accuracy:", accuracy_score(y_valid, pred))
print("log_loss:", log_loss(y_valid, proba, labels=model.classes_))
print(classification_report(y_valid, pred))


accuracy: 0.5869565217391305
log_loss: 0.8636955078889451
              precision    recall  f1-score   support

    away_win       0.42      0.56      0.48        18
        draw       0.38      0.13      0.19        23
    home_win       0.68      0.80      0.74        51

    accuracy                           0.59        92
   macro avg       0.49      0.50      0.47        92
weighted avg       0.55      0.59      0.55        92



# 21단계: 예측 결과 확인

In [114]:
result = valid_df[[
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "target"
]].copy()

result["predicted"] = pred

for i, cls in enumerate(model.classes_):
    result[f"prob_{cls}"] = proba[:, i]

result.head(20)


,date,home_team,away_team,home_score,away_score,target,predicted,prob_away_win,prob_draw,prob_home_win
368,2026-03-30,Uzbekistan,Venezuela,0,0,draw,home_win,0.061408,0.337597,0.600995
369,2026-03-30,New Zealand,Chile,4,1,home_win,away_win,0.515588,0.215574,0.268839
370,2026-03-30,Germany,Ghana,2,1,home_win,home_win,0.046053,0.227869,0.726078
371,2026-03-30,Cape Verde,Finland,1,1,draw,home_win,0.186730,0.338916,0.474354
372,2026-03-31,Austria,South Korea,1,0,home_win,home_win,0.187514,0.302871,0.509615
373,2026-03-31,Algeria,Uruguay,0,0,draw,home_win,0.215209,0.334799,0.449991
374,2026-03-31,South Africa,Panama,1,2,away_win,draw,0.309443,0.377806,0.312752
375,2026-03-31,Argentina,Zambia,5,0,home_win,home_win,0.063099,0.289153,0.647748
376,2026-03-31,Morocco,Paraguay,2,1,home_win,home_win,0.184367,0.212358,0.603275
377,2026-03-31,Australia,Curaçao,5,1,home_win,away_win,0.367329,0.339717,0.292954


# 22단계: 결과 저장

In [115]:
output_path = DATA_DIR / "last1year_transfermarkt_fotmob_recent5_predictions.csv"

result.to_csv(output_path, index=False, encoding="utf-8-sig")

print(output_path)


/Users/minseobeom/Desktop/WorldCup2026/data/processed/last1year_transfermarkt_fotmob_recent5_predictions.csv
